# 🧬 Q4. Viterbi Algorithm – Nature Primer HMM Setup
## 🔧 HMM Configuration

---

### 📌 States (Hidden)
The model uses three hidden states representing regions in a gene sequence:
- `E` — **Exon**
- `5` — **5′ Splice Site**
- `I` — **Intron**

---

### 🧪 Observations (Alphabet)
Possible observed symbols (DNA bases):
A,G,C,T 

---

### 🔁 Transition Probabilities

These define the likelihood of moving from one state to another:

| **From** | **To** | **Probability** |
|----------|--------|-----------------|
| Start    | E      | 1.0             |
| E        | E      | 0.9             |
| E        | 5      | 0.1             |
| 5        | I      | 1.0             |
| I        | I      | 0.9             |
| I        | End    | 0.1             |

---

### 🎯 Emission Probabilities

These define the likelihood of each state emitting a specific DNA base:

| **State** | **A**  | **C**  | **G**  | **T**  |
|-----------|--------|--------|--------|--------|
| `E`       | 0.25   | 0.25   | 0.25   | 0.25   |
| `5`       | 0.05   | 0.00   | 0.95   | 0.00   |
| `I`       | 0.40   | 0.10   | 0.10   | 0.40   |

---

This HMM is designed for gene prediction and recognizes coding (exon), non-coding (intron), and splice junction regions.


In [2]:
import math

def safe_log(value):
    """Returns the natural log of a value, or -inf if the value is 0."""
    return -math.inf if value == 0 else math.log(value)

def compute_log_probability(hidden_path, observed_sequence):
    """
    Computes the log-probability of an observed sequence given a state path
    using transition and emission probabilities.
    """
    if len(hidden_path) != len(observed_sequence):
        raise ValueError("Length of state path and sequence must match.")

    total_log_prob = 0.0
    previous_state = 'Start'

    for i in range(len(observed_sequence)):
        current_state = hidden_path[i]
        observed_symbol = observed_sequence[i]

        trans_prob = hmm_transitions[previous_state][current_state]
        emit_prob = hmm_emissions[current_state][observed_symbol]

        total_log_prob += safe_log(trans_prob) + safe_log(emit_prob)
        previous_state = current_state

    # Handle final transition from 'I' to 'End'
    if previous_state == 'I':
        total_log_prob += safe_log(hmm_transitions[previous_state]['End'])

    return round(total_log_prob, 2)

# HMM Definitions
states_list = ['E', '5', 'I']
hmm_transitions = {
    'Start': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1}
}

hmm_emissions = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.00, 'G': 0.95, 'T': 0.00},
    'I': {'A': 0.40, 'C': 0.10, 'G': 0.10, 'T': 0.40}
}

# Input state path and observed sequence
hidden_states = "EEEEEEEEEEEEEEEEEE5IIIIIII"
observed_dna = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Calculate and print the log-probability
log_probability = compute_log_probability(hidden_states, observed_dna)
print(f"Log probability of the provided path: {log_probability}")


Log probability of the provided path: -41.22


In [3]:
import math

def safe_log(x):
    """Returns log(x), or -inf if x is 0 to avoid math domain errors."""
    return -math.inf if x == 0 else math.log(x)

def run_viterbi(observed_seq):
    sequence_length = len(observed_seq)
    viterbi_table = [{}]  # List of dicts to store max log probabilities
    state_tracker = {}    # To keep track of the optimal path

    # Initialize with starting state 'E'
    for initial_state in ['E']:
        initial_trans = hmm_transitions['Start'][initial_state]
        initial_emit = hmm_emissions[initial_state][observed_seq[0]]
        viterbi_table[0][initial_state] = safe_log(initial_trans) + safe_log(initial_emit)
        state_tracker[initial_state] = [initial_state]

    # Fill in the Viterbi table
    for time_step in range(1, sequence_length):
        viterbi_table.append({})
        updated_tracker = {}

        for current_state in hmm_states:
            best_score = -math.inf
            best_prev_state = None

            for prev_state in viterbi_table[time_step - 1]:
                if current_state in hmm_transitions.get(prev_state, {}):
                    trans_log = safe_log(hmm_transitions[prev_state][current_state])
                    emit_log = safe_log(hmm_emissions[current_state][observed_seq[time_step]])
                    total_score = viterbi_table[time_step - 1][prev_state] + trans_log + emit_log

                    if total_score > best_score:
                        best_score = total_score
                        best_prev_state = prev_state

            if best_prev_state:
                viterbi_table[time_step][current_state] = best_score
                updated_tracker[current_state] = state_tracker[best_prev_state] + [current_state]

        state_tracker = updated_tracker

    # Final state selection
    highest_final_score = -math.inf
    final_best_state = None
    for state in hmm_states:
        score = viterbi_table[sequence_length - 1].get(state, -math.inf)
        if score > highest_final_score:
            highest_final_score = score
            final_best_state = state

    best_state_path = ''.join(state_tracker[final_best_state])
    return best_state_path, round(highest_final_score, 2)

# HMM components
hmm_states = ['E', '5', 'I']
hmm_transitions = {
    'Start': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1}
}
hmm_emissions = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.00, 'G': 0.95, 'T': 0.00},
    'I': {'A': 0.40, 'C': 0.10, 'G': 0.10, 'T': 0.40}
}

# Sample input
dna_input = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Run Viterbi algorithm
optimal_path, log_probability = run_viterbi(dna_input)

print("Most likely path for the given sequence:", optimal_path)
print("Log probability of the best path:", log_probability)


Most likely path for the given sequence: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability of the best path: -38.68
